# From the Auto Quote API to an MCP server

In the previous notebook we wrote an MCP server by hand: two tools in front of a tiny API. The real Auto Quote backend has 23 routes. Do we have to write a tool for each of them?

No. A FastAPI app describes all its routes in an OpenAPI document, which is JSON with a JSON Schema for every input. [FastMCP](https://gofastmcp.com) can read it and generate the tools: `FastMCP.from_fastapi(app)`.

1. Convert the whole app in one line
2. Look at the result: easy does not mean good
3. Launch it and connect Claude
4. Curate: keep what an agent needs and make it readable
5. Use it with an agent, then connect Claude again

This notebook uses the **real app**: importing it opens the files in `backend/data/`, so you see your own catalogue and configuration. It needs the same `.env.local` as the app. If the catalogue is empty, add a few products in the app first.

In [ ]:
# Setup - Can be ignored
import functools
import json
import os
import threading
import time
from pathlib import Path

import uvicorn
from dotenv import load_dotenv
from fastmcp import Client, FastMCP
from fastmcp.server.providers.openapi import MCPType, RouteMap
from openai import OpenAI

MODEL = "gpt-6-luna"  # text and tool calling

# walk up from the working directory to the repo root (the folder with .git)
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())


@functools.cache
def get_client() -> OpenAI:
    load_dotenv(REPO_ROOT / ".env.local")
    return OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL") or None)


client = get_client()

RUNNING_SERVERS: dict[int, tuple[uvicorn.Server, threading.Thread]] = {}


def serve(app, port: int) -> str:
    """Starts an ASGI app in a background thread and returns its base URL. Running a cell twice restarts the server."""
    stop(port)
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    while not server.started:
        time.sleep(0.05)
    RUNNING_SERVERS[port] = (server, thread)
    return f"http://127.0.0.1:{port}"


def stop(port: int) -> None:
    if port in RUNNING_SERVERS:
        server, thread = RUNNING_SERVERS.pop(port)
        server.should_exit = True
        thread.join()

## 1. The app and the one-line conversion

`auto_quote.main` builds the same FastAPI `app` that `make dev` serves. We do not start it as a server: FastMCP calls it directly in the same process, so no port and no network are involved.

In [ ]:
from auto_quote.main import app

routes = [(method.upper(), path, operation["operationId"]) for path, methods in app.openapi()["paths"].items() for method, operation in methods.items()]
print(len(routes), "routes")
for method, path, _ in routes:
    print(f"{method:6} {path}")

In [ ]:
mcp_all = FastMCP.from_fastapi(app, name="auto-quote")


async def show_tools(mcp: FastMCP) -> None:
    async with Client(mcp) as mcp_client:  # a client can talk to a server object directly
        tools = await mcp_client.list_tools()
    print(len(tools), "tools")
    for tool in tools:
        print(f"- {tool.name}: {tool.description}")


await show_tools(mcp_all)

## 2. Easy does not mean good

The conversion worked, and an agent could already use all of this. But look at what we got:

- **Unreadable names**: `get_products_api_v1_products_get` is built from the Python function, the URL and the method. A model chooses tools by name and description.
- **No description**: the routes have no docstring, so the only text is a title like `Get Products`. The model has to guess what a tool is for.
- **Too much**: file uploads, a PDF download and the document file route are not useful for an agent. Every extra tool makes the choice harder.
- **Too powerful**: `PUT`, `POST` and `DELETE` routes become tools too. With them an agent can overwrite the extraction configuration or delete the document index, on its own.

So the generation is the easy part. **Choosing what to expose** is the real work, and it is a decision about what you trust an agent to do.

## 3. Launch it and connect Claude

Before we curate anything, let us see what the raw conversion feels like from the other side. Until now we called the server from inside the notebook. An MCP client such as **Claude Code** is a separate program, so the server has to listen on a port. The cell below starts it in the background, like the servers of the previous notebook, and prints the command that connects Claude Code to it.

> **Careful: this raw server exposes all 23 routes, including the ones that write and delete, on your real data in `backend/data/`.** Claude Code asks before it runs a tool: read the tool name and refuse anything that is not a read (`get_…`). We curate in the next section, and the same URL will then serve the safe version.


In [ ]:
MCP_URL = serve(mcp_all.http_app(), port=8003) + "/mcp"
print("MCP server running on", MCP_URL)
print()
print("In a terminal, bind it to Claude Code:")
print(f"  claude mcp add --transport http auto-quote {MCP_URL}")


Run the printed command in a terminal (not in this notebook), then start a **new** Claude Code session, or run `/mcp` in the current one, and check that `auto-quote` is connected. Ask for example:

- *Which tools do you get from auto-quote?* You see the unreadable names from section 1.
- *Which products do we have in the catalogue?* Claude has to guess which of the tools to use.

The server only runs as long as this notebook's kernel does. To unbind Claude Code later: `claude mcp remove auto-quote`.


## 4. Curate

`from_fastapi` takes three options for this:

- `route_maps`: rules that decide, per method and URL pattern, whether a route becomes a tool or is excluded. The first matching rule wins.
- `mcp_names`: new names, keyed by the `operationId` of the route (the third column printed above).
- `mcp_component_fn`: a hook that runs on every generated tool, here to set a description.

We keep four read-only routes: the product catalogue, the extraction fields, the JSON Schema and prompt that Auto Quote generates from them (the schema from the first notebook), and the document index status.

In [ ]:
EXPOSED_ROUTES = {
    "get_products_api_v1_products_get": (
        "list_products",
        "List every product of the catalogue with its reference, description, unit, unit price in CHF and specifications.",
    ),
    "get_extraction_configuration_api_v1_extraction_configuration_get": (
        "get_extraction_fields",
        "List the fields that are extracted from a customer request: key, label, type, whether it is required and its allowed values.",
    ),
    "preview_extraction_configuration_api_v1_extraction_configuration_preview_get": (
        "get_extraction_schema",
        "Get the JSON Schema and the prompt that are generated from the extraction fields and sent to the language model.",
    ),
    "get_document_index_status_api_v1_documents_index_get": (
        "get_document_index_status",
        "Get the status of the document index that is used to find similar past requests and offers.",
    ),
}


def describe(route, component) -> None:
    component.description = EXPOSED_ROUTES[route.operation_id][1]


exposed_paths = [path for method, path, operation_id in routes if operation_id in EXPOSED_ROUTES]

mcp = FastMCP.from_fastapi(
    app,
    name="auto-quote",
    route_maps=[
        RouteMap(methods=["GET"], pattern="^(" + "|".join(exposed_paths) + ")$", mcp_type=MCPType.TOOL),
        RouteMap(mcp_type=MCPType.EXCLUDE),
    ],
    mcp_names={operation_id: tool_name for operation_id, (tool_name, _) in EXPOSED_ROUTES.items()},
    mcp_component_fn=describe,
)

await show_tools(mcp)

Four tools instead of 23, with names and descriptions a model can work with. The input schemas come from the routes without any extra work: none of these four needs arguments, but a route with a path parameter or a request body would show it in `input_schema`.

Before an agent uses it, we call a tool ourselves, as a client would. The request goes through the real route, the real `CurrentUser` check and the real controller.

In [ ]:
async with Client(mcp) as mcp_client:
    result = await mcp_client.call_tool("list_products")

print(json.dumps(result.data, indent=2, ensure_ascii=False)[:1500])

## 5. Use it with an agent

The same loop as in the previous notebook: the MCP tools go to the model as function tools, the model asks for calls, and we run them on the MCP server.

In [ ]:
def to_openai_tool(tool) -> dict:
    return {"type": "function", "name": tool.name, "description": tool.description, "parameters": tool.input_schema}


async def ask_agent(question: str, max_steps: int = 5) -> str:
    async with Client(mcp) as mcp_client:
        tools = [to_openai_tool(tool) for tool in await mcp_client.list_tools()]
        messages = [{"role": "user", "content": question}]

        for _ in range(max_steps):
            response = client.responses.create(model=MODEL, input=messages, tools=tools)
            messages += response.output
            tool_calls = [item for item in response.output if item.type == "function_call"]
            if not tool_calls:
                return response.output_text

            for call in tool_calls:
                arguments = json.loads(call.arguments)
                print(f"tool call: {call.name}({arguments})")
                result = await mcp_client.call_tool(call.name, arguments, raise_on_error=False)
                output = "\n".join(block.text for block in result.content)
                messages.append({"type": "function_call_output", "call_id": call.call_id, "output": output})

    raise RuntimeError(f"No answer after {max_steps} steps")

In [ ]:
print(await ask_agent("Which fields do we extract from a customer request, and which of them are required?"))

In [ ]:
print(await ask_agent("Which of our products are priced per hour, and what is the cheapest one?"))

The agent answers from the live data of the app. We wrote no tool by hand: the routes, their inputs and the JSON Schema were already there.

### Connecting Claude to the curated server

The curated server replaces the raw one on the same port, so the URL does not change. If you bound Claude Code in section 3, reconnect it (`/mcp` in Claude Code, or start a new session) and it now lists four tools with readable names and descriptions. If you skipped section 3, run the printed command once.


In [ ]:
MCP_URL = serve(mcp.http_app(), port=8003) + "/mcp"
print("Curated MCP server running on", MCP_URL)
print(f"claude mcp add --transport http auto-quote {MCP_URL}")


Things to consider before exposing the app this way outside a workshop:

- **Authentication.** The MCP server calls the app's routes, so the `CurrentUser` check still runs. But the prototype's `SessionCookieProvider` accepts every caller and issues a new session, so on `localhost` it protects nothing. By default FastMCP does not forward credential headers from the MCP client to the API. A shared deployment needs a real `AuthProvider` and a deliberate decision on how the agent identifies itself.
- **Writes.** Adding a `POST` or `PUT` route to `EXPOSED_ROUTES` is a one-line change, which is exactly why it deserves a review. Quotes are sent to customers; an agent that can create or change them is a different risk from one that can read the catalogue.
- **Cost.** Routes that call the language model (such as creating an extraction) cost money on every call.
- **Other APIs.** For an API that is not a FastAPI app in your process (a CRM, an ERP), `FastMCP.from_openapi(spec, client)` does the same from its OpenAPI document.

## Takeaways

- A FastAPI app already describes its routes and their inputs as JSON Schema, so converting it to MCP is one line: `FastMCP.from_fastapi(app)`.
- The generated server is complete but not good: unreadable names, no descriptions, and write routes included.
- The work is in curating: filter the routes (`route_maps`), name the tools (`mcp_names`) and describe them (`mcp_component_fn`). The descriptions are what the model reads.
- What you expose is what an agent may do. Start read-only, and keep authentication and cost in mind.

## Shutting down

The last cell stops the MCP server and every other server started in this notebook. Restarting or closing the kernel does the same. Claude Code is not told: it will show `auto-quote` as disconnected until you remove it with `claude mcp remove auto-quote` in a terminal.


In [ ]:
# Stops the MCP server started in this notebook
for port in list(RUNNING_SERVERS):
    stop(port)
    print("Stopped the server on port", port)
